In [ ]:
import os
import sys
from decimal import Decimal
import warnings
import matplotlib.pyplot as plt
import pandas_ta as ta  # noqa: F401

warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
print(f"Root path: {root_path}")
sys.path.append(root_path)
from core.data_sources.clob import CLOBDataSource
from core.data_structures.candles import Candles

# Import data
- Get candles for the last x days

In [ ]:

# Get trading rules and candles
clob = CLOBDataSource()

In [ ]:
CONNECTOR_NAME = "binance"
INTERVALS = "1h"
trading_pair = "ETH-USDT"
# DAYS = 60

clob.load_candles_cache(root_path)
all_candles = clob.get_candles_from_cache(CONNECTOR_NAME, trading_pair, INTERVALS)
print(all_candles)


In [ ]:
candles: Candles = all_candles
df = candles.data[57000:]
candles.data = candles.data[57000:]

df

In [ ]:
list(df.columns.values)



In [ ]:
import pandas as pd
import numpy as np

# df = candles.data.copy()

# Feature engineering
df["log"] = np.log(df["close"])
df["returns"] = df["log"].diff()  # Alternative to pct_change() of log values
df["range"] = (df["high"] / df["low"]) - 1

# Calculate relative volume
n = 20
df["volume_MA"] = df["volume"].rolling(window=n).mean()

# Ensure we're working with Series, not DataFrames
# This is the key fix for your error
df["relative_volume"] = df["volume"].squeeze() / df["volume_MA"]

# Drop any NaNs from the calculations
df.dropna(inplace=True)

# Final training matrix
X_train = df[["returns", "range", "relative_volume"]]
X_train



In [ ]:
from hmmlearn.hmm import GaussianHMM
import joblib as jl

models_path = os.path.join(root_path, 'data', 'ml_models')

model = GaussianHMM(n_components=4, covariance_type='full', n_iter=1000, random_state=42)
model.fit(X_train)

# Save the trained model as a joblib file
jl.dump(model, filename=f"{models_path}/hmm_gmm_model|{CONNECTOR_NAME}|{trading_pair}|{INTERVALS}.joblib")
print(f"GMM-HMM model saved to: {models_path}")

# print(model.predict_proba(X_train))
regime_probs = model.predict_proba(X_train)
hidden_states = model.predict(X_train)

candles_df = candles.data[n-1:]

# Append states to df
df['regime'] = hidden_states
# candles_df['regime_probs'] = regime_probs

for i in range(regime_probs.shape[1]):
    candles_df[f'regime_prob_{i}'] = regime_probs[:, i]

# candles_df


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.03, 
    subplot_titles=(trading_pair, 'Regime Probabilities'),
    row_heights=[0.7, 0.3]
)

# Add candlestick
fig.add_trace(go.Candlestick(x=candles_df.index,
                             open=candles_df['open'],
                             high=candles_df['high'],
                             low=candles_df['low'],
                             close=candles_df['close'],
                             name='OHLC'),
              row=1, col=1)


regime_colors = ['#FF5733', '#33FF57', '#3357FF', '#F3FF33']  # Choose your colors
for i in range(4):
    fig.add_trace(
        go.Scatter(
            x=candles_df.index,
            y=regime_probs[:, i] if isinstance(regime_probs, np.ndarray) else regime_probs.iloc[:, i],
            mode='lines',
            name=f'Regime {i}',
            line=dict(color=regime_colors[i], width=2)
        ),
        row=2, col=1
    )


# Update layout for dark theme
fig.update_layout(
    title=f'{CONNECTOR_NAME} - {trading_pair} - {INTERVALS}',
    width=1200, height=800,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='Regime Probability', showgrid=False),
    showlegend=True
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
# fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()

### Signal Generation
The `signal` column is generated by evaluating the regime probabilities for each time step. 
For each row, if the highest regime probability exceeds a threshold, the signal is set to the regime number (0–3) with the highest probability. If no regime probability exceeds the threshold, the signal is set to -1, indicating no strong regime detected. 

This approach helps identify periods where the model is confident about a particular regime.

In [ ]:
# Generate signal
threshold = 0.65
candles_df["signal"] = 0

# Find the regime with the highest probability above the threshold
regime_prob_cols = [f'regime_prob_{i}' for i in range(regime_probs.shape[1])]
probs = candles_df[regime_prob_cols]

# For each row, get the regime with the highest probability above threshold, else -1
candles_df["signal"] = probs.apply(
    lambda row: int(row.idxmax()[-1]) if row.max() > threshold else -1, axis=1
)

# candles_df[5000:]

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.02,
    subplot_titles=('OHLC', 'Regime Probabilities', 'Signal'),
    row_heights=[0.6, 0.2, 0.2]
)

# Add candlestick plot
fig.add_trace(
    go.Candlestick(
        x=candles_df.index,
        open=candles_df['open'],
        high=candles_df['high'],
        low=candles_df['low'],
        close=candles_df['close'],
        name='Candlesticks'
    ),
    row=1, col=1
)

# Add regime probabilities as lines
regime_colors = ['#FF5733', '#33FF57', '#3357FF', '#F3FF33']
for i in range(4):
    fig.add_trace(
        go.Scatter(
            x=candles_df.index,
            y=candles_df[f'regime_prob_{i}'],
            mode='lines',
            name=f'Regime {i}',
            line=dict(color=regime_colors[i], width=2)
        ),
        row=2, col=1
    )

# Add the signal line (ranging from -1 to 3)
fig.add_trace(
    go.Scatter(
        x=candles_df.index,
        y=candles_df['signal'],
        mode='lines',
        name='Signal',
        line=dict(color="white")
    ),
    row=3, col=1
)

# Update layout for dark theme
fig.update_layout(
    title=f'{CONNECTOR_NAME} - {trading_pair} - {INTERVALS}',
    width=1200, height=800,
    font=dict(color='#e1e1e1'),
    plot_bgcolor='#1e1e1e',
    paper_bgcolor='#1e1e1e',
    xaxis_rangeslider_visible=False,
    legend=dict(bgcolor='rgba(0,0,0,0)'),
    yaxis=dict(title='Price'),
    yaxis2=dict(title='Regime Probability', showgrid=False),
    yaxis3=dict(title='Signal', showgrid=False, range=[-1.2, 3.2]),
    showlegend=True
)

# Update axes
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='#323232', zeroline=False)

# Show the plot
fig.show()

# CONCLUSION

In this notebook, we have implemented a strategy combining the MACD (Moving Average Convergence Divergence) indicator with Bollinger Bands. We've visualized these indicators along with the price data and generated signals based on their interactions. This approach provides a solid foundation for our trading strategy.
 
## Key components of our strategy include:
 1. MACD for trend identification
 2. Bollinger Bands for volatility measurement and potential reversal points
 3. A signal line derived from the combination of these indicators
 
 The next step is to backtest this strategy to evaluate its profitability and robustness. For this purpose, we have created a controller file named `macd_bb.py` in this folder. This file implements the logic we've developed here, allowing us to conduct comprehensive backtests in the subsequent notebook.